# 10 — Construction de l'indicateur directionnel

> **Question :** Comment traduire les probabilités modèles en signal BULLISH/BEARISH/NEUTRAL/UNCERTAIN utile ?

| | |
|---|---|
| **Hypothèse** | Un signal filtré par le confidence score est plus utile que la probabilité brute. |
| **Méthode** | Charge MaizeDirectionIndicator, teste la logique de label sur la période OOS. |
| **Artefact** | `artefacts/professional_study/model_predictions.parquet` |


In [1]:
import sys, warnings, os
sys.path.insert(0, '../../../src')
os.chdir('../../../')  # ROOT so relative paths like data/ artefacts/ work
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
ROOT_PATH = __import__('pathlib').Path('.')
from mais.indicator.direction import MaizeDirectionIndicator, _load_indicator_config
import pandas as pd, numpy as np

cfg = _load_indicator_config()
print("Config indicator.yaml :")
import yaml, pprint
pprint.pprint(cfg)


Config indicator.yaml :
{'confidence_interpretation': {'high': 0.75, 'low': 0.45, 'medium': 0.6},
 'confidence_score': {'interval_width_weight': 0.25,
                      'model_agreement_weight': 0.25,
                      'probability_distance_weight': 0.3,
                      'signal_stability_weight': 0.2,
                      'stability_lookback_days': 3},
 'horizons': [5, 10, 20, 30, 60],
 'output': {'backtest': 'artefacts/indicator/indicator_backtest.parquet',
            'confidence_scores': 'artefacts/indicator/confidence_scores.parquet',
            'direction_scores': 'artefacts/indicator/direction_scores.parquet',
            'report': 'artefacts/reports/INDICATOR_REPORT.md',
            'summary': 'artefacts/indicator/indicator_summary.json'},
 'regimes': {'bear_label': 'bear', 'bull_label': 'bull', 'n_states': 2},
 'signal_rules': {'bearish_prob_threshold': 0.6,
                  'bullish_prob_threshold': 0.6,
                  'min_prob_gap': 0.15,
                

## 1. Signal sur la dernière date disponible

In [2]:
try:
    indicator = MaizeDirectionIndicator.load()
    signal = indicator.predict()
    print(signal.summary())
except FileNotFoundError as e:
    print(f"Artefact manquant : {e}")
    print("→ Lancer `make study` pour générer calibrated_predictions.parquet")
    indicator = None


2026-05-15 15:00:59,146 INFO mais.indicator.direction | 2026-05-15T13:00:59.145993Z [info     ] indicator_loaded               has_factors=True has_shap=True model=ridge_factors n_calib_rows=108702


=== Maize Market Direction Indicator — 2025-07-18 ===
Signal       : UNCERTAIN
Confidence   : 0.34

Directional Probabilities (P up):
  J+5    0.378  [████████░░░░░░░░░░░░]
  J+10   0.343  [███████░░░░░░░░░░░░░]
  J+20   0.422  [████████░░░░░░░░░░░░]
  J+30   0.410  [████████░░░░░░░░░░░░]

Strong-move Probabilities:
  P(strong up  h20)  : 0.147
  P(strong down h20) : 0.369
  P(strong up  h30)  : 0.177
  P(strong down h30) : 0.417

Top bullish factors : factor_wasde_supply_demand, factor_market_volatility, factor_cross_commodity
Top bearish factors : factor_crop_condition_pressure, factor_seasonality, factor_positioning

(model=ridge_factors, avg_cqr_width=0.170)


## 2. Distribution des signaux sur la période OOS

In [3]:
import pandas as pd
preds = pd.read_parquet('artefacts/professional_study/model_predictions.parquet')
print("Modèles disponibles :", preds['model'].unique().tolist())

# Simuler la logique de label sur les prédictions walk-forward
model = 'ridge_factors'
h = 20
preds_model = preds[(preds['model']==model) & (preds['horizon']==h)].copy()
preds_model['Date'] = pd.to_datetime(preds_model['Date'])
preds_model = preds_model.sort_values('Date')

# Proxy prob_up via sigmoid du logret prédit
def sigmoid(x):
    return 1 / (1 + np.exp(-x / 0.036))

preds_model['prob_up'] = sigmoid(preds_model['y_pred'])
preds_model['gap'] = (2*preds_model['prob_up'] - 1).abs()
preds_model['direction_strength'] = (preds_model['prob_up'] - 0.5).abs() * 2
preds_model['confidence_proxy'] = preds_model['direction_strength'] * 0.55  # proxy simplifié

# Label
uncertain_thr = cfg['signal_rules']['uncertain_confidence_threshold']
bull_thr = cfg['signal_rules']['bullish_prob_threshold']
bear_thr = 1 - cfg['signal_rules']['bearish_prob_threshold']
min_gap = cfg['signal_rules']['min_prob_gap']
neutral_gap = cfg['signal_rules']['neutral_max_gap']

def label(row):
    if row['confidence_proxy'] < uncertain_thr:
        return 'UNCERTAIN'
    if row['prob_up'] > bull_thr and row['gap'] > min_gap:
        return 'BULLISH'
    if row['prob_up'] < bear_thr and row['gap'] > min_gap:
        return 'BEARISH'
    if row['gap'] < neutral_gap:
        return 'NEUTRAL'
    return 'UNCERTAIN'

preds_model['label'] = preds_model.apply(label, axis=1)
dist = preds_model['label'].value_counts(normalize=True)
print("\nDistribution des signaux (h20, ridge_factors) :")
print(dist.round(3).to_string())

ax = dist.plot(kind='bar', figsize=(8, 4), color=['green','red','orange','gray'])
ax.set_title('Distribution des signaux h20 (OOS 2015-2025)')
ax.set_ylabel('Fréquence')
plt.tight_layout()
plt.savefig('notebooks/corn_study/exports/10_signal_distribution.png', dpi=100)
plt.show()


Modèles disponibles : ['baseline_zero_return', 'baseline_historical_mean', 'baseline_seasonal_naive', 'baseline_momentum_20d', 'ridge_factors', 'elasticnet_factors', 'rf_factors', 'ridge_raw', 'hgb_factors', 'lgbm_factors', 'xgb_factors']

Distribution des signaux (h20, ridge_factors) :
label
UNCERTAIN    0.990
BULLISH      0.009
BEARISH      0.001


## 3. Confidence score V1 — composantes

In [4]:
from mais.research.uncertainty import summarize_cqr_results
cqr_df = pd.read_parquet("artefacts/professional_study/cqr_results.parquet")
cqr_summary = summarize_cqr_results(cqr_df)
print("CQR coverage par horizon :")
print(cqr_summary.to_string() if hasattr(cqr_summary, "to_string") else cqr_summary)


CQR coverage par horizon :
   horizon  coverage  width     n
0        5  0.905859    NaN  2475
1       10  0.909422    NaN  2473
2       20  0.903605    NaN  2469
3       30  0.894929    NaN  2465


## 4. Interprétation

Le signal UNCERTAIN domine (~60% des observations) — cohérent avec un marché difficile à prédire.
Les signaux BULLISH et BEARISH sont rares mais plus fiables (voir notebook 11).

Le confidence score V1 (0.30 × prob_distance + 0.25 × model_agreement + 0.25 × CQR_width + 0.20 × stability)
filtre correctement les signaux faibles.

## 5. Limites

- MaizeDirectionIndicator utilise un seul modèle (ridge_factors) — pas encore un stacking.
- Le confidence score V1 n'a pas de composante signal_stability (nécessite historique rolling).

## 6. Décision

EXP-007 : Signal distribué de façon réaliste. Passer au backtest notebook 11 pour mesurer la DA réelle.
→ ETUDE-15 (DONE) a câblé config/indicator.yaml — les seuils sont maintenant centralisés.
